# E0 ResNet50 — External Image Inference and Robustness Testing

**Purpose:** Exploratory evaluation of the trained IP102 E0 model on user-provided images and controlled image transformations. **Not part of the official E0–E4 benchmark.**

**Requirements:** Google Colab, `best_model.pth` in your Google Drive, and your own labeled image files. The notebook does **not** retrain the model. Upload only images you have permission to use. External image filenames are not verified ground-truth labels.

Run cells from top to bottom. **Do not delete earlier cells** when trying new tests. Results are exported to CSV at the end.

In [ ]:
import io
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw
from torchvision import models, transforms
from google.colab import drive, files

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/content/drive/MyDrive/IP102_Proje/results/E0/best_model.pth"
OUTPUT_DIR = Path("/content/e0_robustness_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 1. Load the trained E0 checkpoint

In [ ]:
drive.mount("/content/drive")
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint
model.load_state_dict(state_dict)
model = model.to(DEVICE).eval()
print("E0 checkpoint loaded:", MODEL_PATH)

## 2. Load class names and evaluation preprocessing

The IP102 repository numbers classes from 1 to 102; the model uses indices 0 to 101.

In [ ]:
url = "https://raw.githubusercontent.com/xpwu95/IP102/master/classes.txt"
response = requests.get(url, timeout=30)
response.raise_for_status()
class_names = {}
for line in response.text.splitlines():
    parts = line.strip().split(maxsplit=1)
    if len(parts) == 2 and parts[0].isdigit():
        class_names[int(parts[0]) - 1] = parts[1]
assert set(class_names) == set(range(102)), "Unexpected IP102 class list"

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_topk(image, k=5):
    x = transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        probs = F.softmax(model(x), dim=1)[0].cpu()
    values, indices = torch.topk(probs, k=k)
    return [{"class_id": int(i), "class_name": class_names[int(i)],
             "confidence_pct": float(v) * 100}
            for v, i in zip(values, indices)], probs

print("Loaded", len(class_names), "classes")

## 3. Upload and predict one image

Upload **one** image. For another image, rerun this cell without resetting the model.

In [ ]:
uploaded_single = files.upload()
assert len(uploaded_single) == 1, "Upload exactly one image"
single_filename, single_bytes = next(iter(uploaded_single.items()))
single_image = Image.open(io.BytesIO(single_bytes)).convert("RGB")
display(single_image)
single_top5, _ = predict_topk(single_image)
print("Image:", single_filename)
display(pd.DataFrame(single_top5).round(4))

## 4. Batch prediction on multiple images

Upload multiple images. **Do not interpret filename-based labels as verified ground truth.**

In [ ]:
uploaded_batch = files.upload()
batch_rows = []
for filename, image_bytes in uploaded_batch.items():
    try:
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        preds, _ = predict_topk(img, k=3)
        row = {"filename": filename}
        for rank, p in enumerate(preds, 1):
            row[f"top{rank}_class_id"] = p["class_id"]
            row[f"top{rank}_class"] = p["class_name"]
            row[f"top{rank}_confidence_pct"] = p["confidence_pct"]
        batch_rows.append(row)
    except Exception as exc:
        print(f"Skipped {filename}: {exc}")
batch_df = pd.DataFrame(batch_rows)
display(batch_df.round(3))
batch_df.to_csv(OUTPUT_DIR / "external_image_predictions.csv", index=False)

## 5. Robustness test: image transformations

Upload **one** image. Enter its **verified IP102 class ID** (0–101), or leave `EXPECTED_CLASS_ID = None` if the label is not verified. The reported `matches_expected` field is not a scientific accuracy metric for unverified images.

**Caution:** Center cropping, resizing, padding and occlusion can change the visible subject; results are exploratory. Image transformations are not independent test samples.

In [ ]:
EXPECTED_CLASS_ID = 101  # Change to a VERIFIED class ID, or set None
robust_upload = files.upload()
assert len(robust_upload) == 1, "Upload exactly one image"
robust_filename, robust_bytes = next(iter(robust_upload.items()))
original = Image.open(io.BytesIO(robust_bytes)).convert("RGB")

def pixelate(img, scale=0.03):
    w, h = img.size
    small = img.resize((max(1, round(w*scale)), max(1, round(h*scale))), Image.Resampling.BILINEAR)
    return small.resize((w, h), Image.Resampling.NEAREST)

def occlude(img, fraction):
    out = img.copy()
    w, h = out.size
    bw, bh = round(w*fraction), round(h*fraction)
    x, y = (w-bw)//2, (h-bh)//2
    ImageDraw.Draw(out).rectangle((x, y, x+bw, y+bh), fill=(0, 0, 0))
    return out

def low_resolution(img, size=16):
    return img.resize((size, size), Image.Resampling.BILINEAR).resize(img.size, Image.Resampling.BILINEAR)

variants = {
    "Original": original,
    "Slight blur": original.filter(ImageFilter.GaussianBlur(radius=2)),
    "Strong blur": original.filter(ImageFilter.GaussianBlur(radius=6)),
    "Dark image": ImageEnhance.Brightness(original).enhance(0.3),
    "Bright image": ImageEnhance.Brightness(original).enhance(1.8),
    "Low contrast": ImageEnhance.Contrast(original).enhance(0.3),
    "Rotated 90": original.rotate(90, expand=True),
    "Rotated 180": original.rotate(180, expand=True),
    "Extreme blur": original.filter(ImageFilter.GaussianBlur(radius=15)),
    "Pixelated": pixelate(original),
    "Low resolution": low_resolution(original),
    "40% occlusion": occlude(original, 0.4),
    "70% occlusion": occlude(original, 0.7),
}

robust_rows = []
for name, img in variants.items():
    preds, probs = predict_topk(img, k=1)
    p = preds[0]
    robust_rows.append({
        "filename": robust_filename, "variation": name,
        "predicted_class_id": p["class_id"], "predicted_class": p["class_name"],
        "top1_confidence_pct": p["confidence_pct"],
        "expected_class_id": EXPECTED_CLASS_ID,
        "expected_class_probability_pct": float(probs[EXPECTED_CLASS_ID])*100 if EXPECTED_CLASS_ID is not None else np.nan,
        "matches_expected": p["class_id"] == EXPECTED_CLASS_ID if EXPECTED_CLASS_ID is not None else None,
    })
robust_df = pd.DataFrame(robust_rows)
display(robust_df.round(3))
robust_df.to_csv(OUTPUT_DIR / "transformation_results.csv", index=False)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, (name, img) in zip(axes.flat, variants.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
for ax in axes.flat[len(variants):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "transformation_grid.png", dpi=150)
plt.show()

## 6. Rotation sensitivity (0° to 345°, 15° increments)

Uses the image from section 5. Rotation with `expand=True` also changes geometry before preprocessing, so effects cannot be attributed to orientation alone.

In [ ]:
rotation_rows = []
for angle in range(0, 360, 15):
    img = original.rotate(angle, expand=True, fillcolor=(128, 128, 128))
    preds, probs = predict_topk(img, k=1)
    p = preds[0]
    rotation_rows.append({
        "filename": robust_filename, "angle_deg": angle,
        "predicted_class_id": p["class_id"], "predicted_class": p["class_name"],
        "top1_confidence_pct": p["confidence_pct"],
        "expected_class_id": EXPECTED_CLASS_ID,
        "expected_class_probability_pct": float(probs[EXPECTED_CLASS_ID])*100 if EXPECTED_CLASS_ID is not None else np.nan,
        "matches_expected": p["class_id"] == EXPECTED_CLASS_ID if EXPECTED_CLASS_ID is not None else None,
    })
rotation_df = pd.DataFrame(rotation_rows)
display(rotation_df.round(3))
rotation_df.to_csv(OUTPUT_DIR / "rotation_sensitivity.csv", index=False)

plt.figure(figsize=(11, 5))
if EXPECTED_CLASS_ID is not None:
    plt.plot(rotation_df["angle_deg"], rotation_df["expected_class_probability_pct"], marker="o", label=f"Class {EXPECTED_CLASS_ID} probability")
else:
    plt.plot(rotation_df["angle_deg"], rotation_df["top1_confidence_pct"], marker="o", label="Top-1 confidence")
plt.xlabel("Rotation angle (degrees)")
plt.ylabel("Probability (%)")
plt.xticks(range(0, 360, 30))
plt.ylim(0, 105)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rotation_sensitivity.png", dpi=150)
plt.show()

## 7. Download exported results

Download a ZIP containing all available CSV and PNG files. Store only appropriately licensed images in the repository; model checkpoints should remain in Drive.

In [ ]:
import shutil
archive = shutil.make_archive("/content/e0_robustness_outputs", "zip", OUTPUT_DIR)
print("Exported:", [p.name for p in OUTPUT_DIR.iterdir()])
files.download(archive)

## Reproducibility and interpretation notes

- **E0 baseline:** IP102 test accuracy ≈ 70.81%, macro-F1 ≈ 0.6335, tail recall ≈ 0.5303, ECE ≈ 0.1197. These official test metrics are **not** recalculated by this notebook.
- **External images:** Their true class must be verified independently; filenames alone are not labels.
- **Softmax confidence:** A displayed 100.00% is a rounded model output, not proof of correctness.
- **Experimental scope:** Transformations are exploratory stress tests on selected images, not a replacement for an independent labeled test set.
- **Checkpoint:** Keep `best_model.pth` in Google Drive, outside GitHub.
